In [9]:
!pip install langchain langchain-community sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googl

In [12]:
import pandas as pd

df = pd.read_csv("export_final.csv")
df.head(10)

,article_id,infraction_desc,amende_fixe,points_retrait,categorie_vehicule,type_infraction,mots_cles,cluster,categorie_cluster
0,1,1 املاده ال يجوز الي شخص ان يسوق مركبه ذات محر...,NaN,NaN,leger,definition,documents,3,regles_circulation
1,2,2 املاده استثناء من احكام املاده االولا اعاله ...,NaN,NaN,leger,definition,documents,0,permis_conduite
2,3,3 املاده يجب علا السائقين الحاصلين علا رخصه سي...,NaN,NaN,leger,obligation,documents,2,permissions_exceptions
3,4,4 املاده في حاله السير الدولي ووفقا لالتفاقيه ...,NaN,NaN,leger,definition,"depassement, documents",0,permis_conduite
4,5,5 املاده استثناء من احكام املاده االولا اعاله،...,NaN,NaN,leger,definition,"alcool, documents",0,permis_conduite
5,6,6 املاده ال يجوز الي كان سياقه مركبه فالحيه ذا...,NaN,NaN,leger,definition,documents,3,regles_circulation
6,7,7 املاده .يحدد صنف رخصه السياقه حسب صنف او اصن...,NaN,NaN,leger,definition,"depassement, documents",0,permis_conduite
7,8,8 املاده ال يسمح كل صنف من اصناف رخصه السياقه ...,NaN,NaN,leger,definition,documents,0,permis_conduite
8,9,9 املاده يجب االدالء برخصه السياقه او بالوثيقه...,NaN,NaN,leger,obligation,documents,0,permis_conduite
9,10,10 املاده تسلم رخصه السياقه الا املترشح بعد اج...,NaN,NaN,leger,definition,documents,0,permis_conduite


In [14]:
from langchain_core.documents import Document

documents = []

for index, row in df.iterrows():
  texte = f"حسب المادة {row.article_id} المخالفة هي كذا{row.type_infraction}"
  doc = Document(page_content=texte)
  documents.append(doc)

print(len(documents))

336


In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)
print(len(chunks))


336


In [20]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
texts = [chunk.page_content for chunk in chunks]
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
embeddings = embedding_model.encode(texts)
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))
print(index.ntotal)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

336


In [22]:
def retrieve(query, k=3):
  query_vec = embedding_model.encode([query])
  distances, indices = index.search(np.array(query_vec), k)
  return [texts[i] for i in indices[0]]

question = "ما هي غرامة التوقف غير القانوني؟"
answer = retrieve(question)
print(answer)


['حسب المادة 261 المخالفة هي كذاsanction', 'حسب المادة 259 المخالفة هي كذاsanction', 'حسب المادة 263 المخالفة هي كذاsanction']


In [23]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [30]:
def rag_answer(query):
  docs = retrieve(query)
  context = "\n".join(docs)
  prompt = f"""
        أنت خبير في قوانين مدونة السير المغربية.
    استخدم فقط السياق أدناه للإجابة على السؤال.

    السياق:
    {context}

    السؤال:
    {query}

    الجواب:
      """
  output = generator(
        prompt,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.3,
        return_full_text=False
    )

  return output[0]["generated_text"].strip()

question = "ما هي غرامة التوقف غير القانوني؟"
reponse = rag_answer(question)
print("\n=== جواب الذكاء الاصطناعي ===")
print(reponse)

Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== جواب الذكاء الاصطناعي ===
- افتراضية
       - 2 سنوات
       - 3 أشهر
       - 4 سنوات
       - 6 أشهر
       - 8 أشهر
       - 10 سنوات
       - 12 عاماً

    الإجابة الصحيحة: 10 سنوات

لقد تم تصحيح هذا السؤال وفقًا لنصائح الأسئلة. إذا كنت بحاجة إلى مزيد من المساعدة، فلا تتردد في طرح سؤالك مرة أخرى. 

أعتذر عن الالتباس، لكنني لم أتمكن من تقديم إجابة صحيحة لهذا السؤال. شكراً لك على الرد! 

أعتذر عن ال


In [31]:
!pip install gradio

In [32]:
import gradio as gr

# 1. On crée une fonction simplifiée pour Gradio
def interface_rag(question):
    # On appelle notre fonction RAG qui fait déjà tout le travail
    reponse = rag_answer(question)
    return reponse

# 2. On configure l'interface Web (très simple et élégante)
interface = gr.Interface(
    fn=interface_rag, # La fonction à appeler
    inputs=gr.Textbox(lines=2, placeholder="...اطرح سؤالك حول مدونة السير هنا", label="Votre question :"),
    outputs=gr.Textbox(label="Réponse de l'Assistant Juridique :"),
    title="🚦 Assistant Intelligent - Code de la Route Marocain",
    description="Ce système utilise l'IA (RAG) pour lire les articles du Code de la Route et répondre à vos questions en langage naturel.",
    theme="default",
    examples=[["ما هي غرامة التوقف غير القانوني؟"],
        ["كم عدد النقط التي تخصم في حالة عدم احترام الوقوف المفروض؟"]
    ]
)

# 3. Lancement de l'application Web !
print("Lancement de l'interface Web...")
interface.launch(share=True) # share=True génère un lien public utilisable par tout le monde !

Lancement de l'interface Web...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://060f1ba2ebc9076db9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
